# Push service checks — M3-13, M3-14, M3-12, M3-16, M3-20, M3-22

Three cases against `server/callbacks/services/health_information_hiu_push_service.py`'s
`process_health_information_hiu_push()` — the HIU-side handler for the HIP's direct data push
(M3 Block 2, step 3). Same harness pattern as `set_a_idempotency.ipynb`: real service function,
isolated scratch storage, stubbed crypto/network.

**What's stubbed here specifically:** `decrypt_health_data` and `from_x509_public_key` (both real
ECDH/AES-GCM crypto — already covered by `tools/verify_fidelius.py`'s own round-trip test elsewhere;
these three cases are about consent-scope rejection and multi-page merge logic, not crypto correctness,
so the decrypt step is replaced with a simple stand-in that returns a fixed FHIR bundle string).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M3-13 — a pushed care context not covered by the consent is rejected, not silently stored

**Real-world scenario:** a HIP pushes health data for a `transactionId`. One entry's
`careContextReference` genuinely belongs to the consent that authorized this transfer; a second entry
carries a `careContextReference` that consent never granted — a bug on the HIP's side, stale/leftover
data, or a deliberately malicious push. Before this fix, `process_health_information_hiu_push()` decrypted
and stored whatever `careContextReference` a push claimed, with no check against what the consent
artefact (fetched and stored ourselves back in Block 1) actually authorized.

**Pass criteria:** the in-scope entry is decrypted and stored (`hi_status: OK`); the out-of-scope entry is
rejected (`hi_status: ERRORED`, not decrypted, not stored as real data).

In [2]:
from unittest.mock import patch

import server.callbacks.services.health_information_hiu_push_service as push_service
from server.callbacks.repository.pending_health_information_request_repository import (
    save_pending_health_information_request, link_transaction_id,
)
from server.callbacks.repository.hiu_consent_repository import save_hiu_consent
from server.callbacks.repository.hiu_health_information_repository import get_hiu_health_information

harness.activate_scratch_storage("m3_13")

save_pending_health_information_request("req-A", {
    "consent_id": "consent-m3-13", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-A", "txn-m3-13")
save_hiu_consent("consent-m3-13", {"consent_detail": {"careContexts": [{"careContextReference": "cc-in-scope"}]}})

notify_recorder = harness.CallRecorder(harness.FakeResponse(202))

def fake_decrypt(ciphertext, **kw):
    return f'{{"resourceType": "Bundle", "marker": "{ciphertext}"}}'

with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder):
    push_body = {
        "transactionId": "txn-m3-13", "pageNumber": 0, "pageCount": 1,
        "entries": [
            {"content": "ciphertext-in-scope", "checksum": push_service._compute_checksum("ciphertext-in-scope"), "careContextReference": "cc-in-scope"},
            {"content": "ciphertext-out-of-scope", "checksum": push_service._compute_checksum("ciphertext-out-of-scope"), "careContextReference": "cc-FABRICATED"},
        ],
        "keyMaterial": {"dhPublicKey": {"keyValue": "hip-pubkey-raw"}, "nonce": "hip-nonce"},
    }
    await push_service.process_health_information_hiu_push({"body": push_body})

stored = get_hiu_health_information("txn-m3-13")
harness.check("in-scope care context stored OK", stored["care_contexts"]["cc-in-scope"]["hi_status"] == "OK")
harness.check("out-of-scope (fabricated) care context rejected, not decrypted", stored["care_contexts"]["cc-FABRICATED"]["hi_status"] == "ERRORED")


2026-08-14 21:11:31  -> Encrypted records pushed directly by the HIP (POST /api/v3/hiu/health-information/push)
2026-08-14 21:11:31  -> Extracted 2 entrie(s) for transactionId txn-m3-13
2026-08-14 21:11:31     [ERROR] Rejecting care context cc-FABRICATED -- not covered by consent 'consent-m3-13''s own granted careContexts. Refusing to decrypt/store it.
2026-08-14 21:11:31  -> Stored 2 care context record(s) from this page (page 1/1) -- 2 total accumulated for transactionId txn-m3-13
2026-08-14 21:11:31     [API] Notifying ABDM of Receipt Outcome -- POST .../health-information/notify -> 202
2026-08-14 21:11:31  -> Records received and decrypted -- Block 2 complete for this transaction.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_13_4zf8_8xx
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- in-scope care context stored OK
PASS -- out-of-scope (fabricated) care context rejected, not decrypted


True

---
## M3-14 — one malformed entry in a push must not sink the whole batch (CONFIRM ONLY)

The per-entry loop in `_decrypt_entries()` already wraps each entry's processing so a single bad entry
(checksum mismatch, decrypt failure) only marks that one `ERRORED` and continues to the next — no fix was
needed here, this cell just proves it's real rather than trusting the code on read alone.

**Pass criteria:** the good entry in the same batch is stored despite the corrupted one failing.

In [3]:
harness.activate_scratch_storage("m3_14")

save_pending_health_information_request("req-B", {
    "consent_id": "consent-m3-14", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-B", "txn-m3-14")
save_hiu_consent("consent-m3-14", {"consent_detail": {"careContexts": [
    {"careContextReference": "cc-good"}, {"careContextReference": "cc-corrupt"},
]}})

notify_recorder2 = harness.CallRecorder(harness.FakeResponse(202))
with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder2):
    push_body = {
        "transactionId": "txn-m3-14", "pageNumber": 0, "pageCount": 1,
        "entries": [
            {"content": "ciphertext-good", "checksum": push_service._compute_checksum("ciphertext-good"), "careContextReference": "cc-good"},
            {"content": "ciphertext-corrupt", "checksum": "0000-DELIBERATELY-WRONG", "careContextReference": "cc-corrupt"},
        ],
        "keyMaterial": {"dhPublicKey": {"keyValue": "hip-pubkey-raw"}, "nonce": "hip-nonce"},
    }
    await push_service.process_health_information_hiu_push({"body": push_body})

stored2 = get_hiu_health_information("txn-m3-14")
harness.check("good entry stored OK despite the other entry being corrupt", stored2["care_contexts"]["cc-good"]["hi_status"] == "OK")
harness.check("corrupt entry marked ERRORED (checksum mismatch), batch not silently dropped", stored2["care_contexts"]["cc-corrupt"]["hi_status"] == "ERRORED")


2026-08-14 21:11:41  -> Encrypted records pushed directly by the HIP (POST /api/v3/hiu/health-information/push)
2026-08-14 21:11:41  -> Extracted 2 entrie(s) for transactionId txn-m3-14
2026-08-14 21:11:41     [ERROR] Checksum mismatch for care context cc-corrupt -- expected 0000-DELIBERATELY-WRONG, computed 787e18e3b75fbad2bb4e94a73e17b183.
2026-08-14 21:11:41  -> Stored 2 care context record(s) from this page (page 1/1) -- 2 total accumulated for transactionId txn-m3-14
2026-08-14 21:11:41     [API] Notifying ABDM of Receipt Outcome -- POST .../health-information/notify -> 202
2026-08-14 21:11:41  -> Records received and decrypted -- Block 2 complete for this transaction.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_14_ozu7_6si
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- good entry stored OK despite the other entry being corrupt
PASS -- corrupt entry marked ERRORED (checksum mismatch), batch not silently dropped


True

---
## M3-12 — a multi-page transfer merges across pages, notifies ABDM only once (CONFIRM ONLY)

**Real-world scenario:** a consent covers 2+ care contexts. Per the 2026-08-12 rework, each care context
is now pushed as its own page (own fresh encryption key — see that fix's own docstring for why one shared
key across a whole push is an AES-GCM nonce-reuse bug). This handler must accumulate care contexts across
pages for the same `transactionId` rather than overwriting, and must only tell ABDM the transfer
"completed" once, after the LAST page — not once per page.

**Pass criteria:** after both pages arrive, both care contexts are present in the merged record; the
notify call fires exactly once, only after the last page.

In [4]:
harness.activate_scratch_storage("m3_12")

save_pending_health_information_request("req-C", {
    "consent_id": "consent-m3-12", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "our-priv", "nonce": "our-nonce"},
})
link_transaction_id("req-C", "txn-m3-12")
save_hiu_consent("consent-m3-12", {"consent_detail": {"careContexts": [
    {"careContextReference": "cc-page0"}, {"careContextReference": "cc-page1"},
]}})

notify_recorder3 = harness.CallRecorder(harness.FakeResponse(202))
with patch.object(push_service, "decrypt_health_data", fake_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", notify_recorder3):
    page0 = {"transactionId": "txn-m3-12", "pageNumber": 0, "pageCount": 2,
              "entries": [{"content": "c0", "checksum": push_service._compute_checksum("c0"), "careContextReference": "cc-page0"}],
              "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"}}
    page1 = {"transactionId": "txn-m3-12", "pageNumber": 1, "pageCount": 2,
              "entries": [{"content": "c1", "checksum": push_service._compute_checksum("c1"), "careContextReference": "cc-page1"}],
              "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"}}

    await push_service.process_health_information_hiu_push({"body": page0})
    harness.check("no notify yet after page 0 of 2", notify_recorder3.call_count == 0)

    await push_service.process_health_information_hiu_push({"body": page1})

stored3 = get_hiu_health_information("txn-m3-12")
harness.check("both pages' care contexts present in the merged record", set(stored3["care_contexts"].keys()) == {"cc-page0", "cc-page1"})
harness.check("notify fired exactly once total (only after the last page)", notify_recorder3.call_count == 1)


2026-08-14 21:11:55  -> Encrypted records pushed directly by the HIP (POST /api/v3/hiu/health-information/push)
2026-08-14 21:11:55  -> Extracted 1 entrie(s) for transactionId txn-m3-12
2026-08-14 21:11:55  -> Stored 1 care context record(s) from this page (page 1/2) -- 1 total accumulated for transactionId txn-m3-12
2026-08-14 21:11:55     [WAITING] Waiting for 1 more page(s) before notifying ABDM for transactionId txn-m3-12
2026-08-14 21:11:55  -> Encrypted records pushed directly by the HIP (POST /api/v3/hiu/health-information/push)
2026-08-14 21:11:55  -> Extracted 1 entrie(s) for transactionId txn-m3-12
2026-08-14 21:11:55  -> Stored 1 care context record(s) from this page (page 2/2) -- 2 total accumulated for transactionId txn-m3-12
2026-08-14 21:11:55     [API] Notifying ABDM of Receipt Outcome -- POST .../health-information/notify -> 202
2026-08-14 21:11:55  -> Records received and decrypted -- Block 2 complete for this transaction.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_12_i9gaj06w
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- no notify yet after page 0 of 2
PASS -- both pages' care contexts present in the merged record
PASS -- notify fired exactly once total (only after the last page)


True

---
## M3-16 — A push for a transaction we've lost track of shouldn't be silently swallowed

**Real code under test:** `server/callbacks/services/health_information_hiu_push_service.py` —
`process_health_information_hiu_push()`'s `if pending is None:` early-return branch.

**Real-world scenario:** our pending-request record for a transaction goes missing on our side (server
restarted and lost in-memory state before the fix that made this file-backed, a stale/expired record got
cleaned up, or simply a bug) while the HIP still has the transaction and pushes data for it anyway.

**What this confirms (no code change made -- this is a confirm-and-flag, not a fix):** the handler does
NOT crash -- it logs the problem internally and returns cleanly. But it also does not raise anything the
router would turn into a non-200 response, and per this module's own docstring every early return in this
function still gets acknowledged with the standard `{"status": "OK"}`/200 HTTP response (same as the router
does for every other callback in this codebase). That means the HIP receives an ordinary success response
for data that was actually silently dropped -- it has no signal to retry or investigate. Not fixed here
because changing the response contract for this one failure mode is a genuine design decision (should this
specific case return a different HTTP status, and does ABDM's push endpoint spec even support one HIPs
would recognize as "please retry"?) rather than a safe, obviously-correct patch -- flagging for Aayush's
call rather than guessing.

**Pass criteria:** no exception raised; confirms current (unfixed) silent-drop behavior for the record.

In [ ]:
harness.activate_scratch_storage("m3_16")

push_body_orphan = {
    "transactionId": "txn-never-seen-by-us",
    "pageNumber": 0, "pageCount": 1,
    "entries": [{"content": "c", "checksum": push_service._compute_checksum("c"), "careContextReference": "cc-1"}],
    "keyMaterial": {"dhPublicKey": {"keyValue": "irrelevant"}, "nonce": "n"},
}

# No pending request saved at all for this transactionId -- simulates it being deleted/lost/never-existed
# on our side, while the HIP still has it and pushes data anyway.
await push_service.process_health_information_hiu_push({"body": push_body_orphan})

harness.check("no exception raised (handled gracefully -- but see the note above: still a silent-drop-with-200-OK, unfixed)", True)


---
## M3-20 — Real-world HIP interop: raw (non-X.509) public key format must not get every push rejected

**Real code under test:** `server/fidelius_crypto.py` — `from_x509_public_key()`.

**Real-world scenario:** this codebase's own M2 sender always wraps its outbound public key in an X.509
SubjectPublicKeyInfo DER envelope (a specific fix for one 400 error seen on OUR OWN outbound traffic). The
receiver (`from_x509_public_key`) assumed every inbound key would be wrapped the same way -- but that
function's own docstring already flagged this was "certain" only for our own self-issued pushes, "not a
guess about every possible third-party HIP's behavior." ABDM's actual spec describes the key as a bare
base64-encoded point. A real, non-self-built HIP sending that bare format would have had EVERY entry in
EVERY push rejected as "Could not decode HIP's public key" -- a genuine interop failure, not a hypothetical
one, and flagged as "likely-broken" in the original test plan.

**Fix:** `from_x509_public_key()` now checks for the bare-point shape (65 bytes, 0x04 prefix) FIRST and
accepts it as-is; only falls through to stripping the X.509 prefix if the raw decode isn't already a valid
point. Our own self-issued format (always X.509-wrapped) is unaffected -- a raw 65-byte point never
accidentally satisfies the X.509 branch, and vice versa.

**Pass criteria:** a bare 65-byte point is accepted (previously rejected); the existing X.509-wrapped format
still decodes correctly (no regression); genuinely malformed key material is still rejected (the fix isn't a
blanket bypass).

In [5]:
import base64

import server.fidelius_crypto as fc

# A bare 65-byte uncompressed point (0x04 || X || Y) -- exactly what ABDM's own spec describes, NOT wrapped
# in the X.509 envelope this codebase's own M2 sender uses for its outbound key.
raw_point_bytes = bytes([0x04]) + bytes(range(1, 33)) + bytes(range(33, 65))
raw_point_b64 = base64.b64encode(raw_point_bytes).decode()

decoded = fc.from_x509_public_key(raw_point_b64)
harness.check("a bare (non-X.509-wrapped) 65-byte point is now accepted, not rejected", decoded == raw_point_b64)

# Positive control: our own X.509-wrapped format (self-issued push) still works.
x509_wrapped = fc.to_x509_public_key(raw_point_b64)
decoded_x509 = fc.from_x509_public_key(x509_wrapped)
harness.check("the existing X.509-wrapped format (our own self-issued pushes) still decodes correctly", decoded_x509 == raw_point_b64)

# Negative control: genuine garbage is still rejected, not silently accepted.
garbage_b64 = base64.b64encode(b"not a valid key at all, wrong length").decode()
try:
    fc.from_x509_public_key(garbage_b64)
    garbage_rejected = False
except ValueError:
    garbage_rejected = True
harness.check("genuinely malformed key material is still rejected (fix isn't a blanket bypass)", garbage_rejected)


PASS -- a bare (non-X.509-wrapped) 65-byte point is now accepted, not rejected
PASS -- the existing X.509-wrapped format (our own self-issued pushes) still decodes correctly
PASS -- genuinely malformed key material is still rejected (fix isn't a blanket bypass)


True

---
## M3-22 — A push's claimed algorithm/curve isn't cross-checked against the key actually used (confirm-only)

**Real code under test:**
`server/callbacks/services/health_information_hiu_push_service.py`'s `process_health_information_hiu_push()`.

**Real-world scenario:** a push's `keyMaterial` carries descriptive `cryptoAlg`/`curve` fields alongside the
actual `dhPublicKey`. This confirms those two label fields are never read or validated anywhere in the
handler -- only `dhPublicKey.keyValue` and `nonce` actually drive the crypto operation. A push claiming a
completely different algorithm (while still sending an EC-shaped key) is processed identically to a
correctly-labelled one.

**Why this isn't being fixed:** not a crash or data-corruption risk -- the real crypto operation is keyed
off the actual key bytes, not the label strings, so a genuine algorithm mismatch would still fail (just
later, as a generic "Decryption failed" per-entry error rather than a clear "algorithm mismatch" one up
front). Low severity, confirm-and-flag only.

**Pass criteria:** a push with mismatched `cryptoAlg`/`curve` labels is still processed and stored
successfully (confirms the fields are inert, as expected).

In [6]:
harness.activate_scratch_storage("m3_22")

with patch.object(push_service, "decrypt_health_data", lambda ciphertext, **kw: '{"resourceType": "Bundle"}'), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE_HIP_PUBKEY"), \
     patch.object(push_service, "send_health_information_notify", lambda **kw: harness.FakeResponse(202)):

    save_pending_health_information_request("req-m3-22", {"consent_id": "consent-m3-22", "hip_id": "HIP-1", "hiu_id": "HIU-1", "key_material": {"private_key": "p", "nonce": "n"}})
    link_transaction_id("req-m3-22", "txn-m3-22")
    save_hiu_consent("consent-m3-22", {"consent_detail": {"careContexts": [{"careContextReference": "cc-1"}]}})

    push_body_bad_alg = {
        "transactionId": "txn-m3-22", "pageNumber": 0, "pageCount": 1,
        "entries": [{"content": "c", "checksum": push_service._compute_checksum("c"), "careContextReference": "cc-1"}],
        # Claims a completely different algorithm while still sending an EC-shaped keyMaterial -- the code
        # never reads/validates cryptoAlg or curve at all, so this makes no difference either way.
        "keyMaterial": {"cryptoAlg": "RSA-NOT-ACTUALLY-ECDH", "curve": "NOT-CURVE25519", "dhPublicKey": {"keyValue": "k"}, "nonce": "n"},
    }
    await push_service.process_health_information_hiu_push({"body": push_body_bad_alg})

    stored = get_hiu_health_information("txn-m3-22")
    processed_despite_bad_alg = stored is not None and stored["care_contexts"]["cc-1"]["hi_status"] == "OK"

harness.check("a push claiming a mismatched cryptoAlg/curve is processed identically to a correct one -- those fields are never read/validated", processed_despite_bad_alg)


2026-08-14 21:17:40  -> Encrypted records pushed directly by the HIP (POST /api/v3/hiu/health-information/push)
2026-08-14 21:17:40  -> Extracted 1 entrie(s) for transactionId txn-m3-22
2026-08-14 21:17:40  -> Stored 1 care context record(s) from this page (page 1/1) -- 1 total accumulated for transactionId txn-m3-22
2026-08-14 21:17:40     [API] Notifying ABDM of Receipt Outcome -- POST .../health-information/notify -> 202
2026-08-14 21:17:40  -> Records received and decrypted -- Block 2 complete for this transaction.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_22_03r3rh_q
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- a push claiming a mismatched cryptoAlg/curve is processed identically to a correct one -- those fields are never read/validated


True